In [54]:
import pandas as pd
import numpy as np


import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.neighbors import radius_neighbors_graph
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import k_means
from fcmeans import FCM
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

pio.renderers.default = 'vscode'

In [55]:
data = pd.read_csv("data/XYZ.csv", sep=";", )
print(data.head())
print(f"\nВсего точек N = {len(data)}")

          x         y         z
0  0.183302 -1.348692  0.981512
1  0.463643  1.099327 -0.043228
2  0.957828 -0.074486  0.126483
3 -2.282350 -0.082933 -0.550586
4  0.101040  0.099878 -0.147625

Всего точек N = 50000


In [ ]:
fig = px.scatter_3d(data, x='x', y='y', z='z',
                    title='3D диаграмма рассеяния всех точек',
                    opacity=0.6)
fig.update_traces(marker=dict(size=3, color='blue'))
fig.update_layout(width=900, height=700)
fig.show()

In [ ]:
X = data[['x', 'y', 'z']].values

eps = 0.0775
min_samples = 5

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
labels = dbscan.fit_predict(X)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = np.sum(labels == -1)

print(f"eps={eps}, min_samples={min_samples}")
print(f"Кластеров: {n_clusters}")
print(f"Шума: {n_noise}")

eps=0.0775, min_samples=5
Кластеров: 5
Шума: 1382


In [ ]:
data['cluster'] = labels.astype(str)

fig = px.scatter_3d(data, x='x', y='y', z='z',
                    color='cluster',
                    title=f'Результат DBSCAN кластеризации (eps={eps}, кластеров={n_clusters})',
                    opacity=0.7,
                    color_discrete_sequence=px.colors.qualitative.Set1)
fig.update_traces(marker=dict(size=3))
fig.update_layout(width=900, height=700)
fig.show()

In [ ]:
X = data[['x', 'y', 'z']].values
N = len(X)

# Перебираем разные eps
eps_values = np.arange(0.07, 0.10, 0.0025)
clusters_count = []

for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    clusters_count.append(n_clusters)
    print(f"eps={eps:.2f} -> кластеров: {n_clusters}")

# Находим eps для 5 кластеров
eps_for_5 = None
for eps, n_clust in zip(eps_values, clusters_count):
    if n_clust == 5:
        eps_for_5 = eps
        break

print(f"\nДля 5 кластеров нужно eps = {eps_for_5}")

eps=0.07 -> кластеров: 16
eps=0.07 -> кластеров: 14
eps=0.08 -> кластеров: 8
eps=0.08 -> кластеров: 5
eps=0.08 -> кластеров: 5
eps=0.08 -> кластеров: 4
eps=0.09 -> кластеров: 3
eps=0.09 -> кластеров: 3
eps=0.09 -> кластеров: 3
eps=0.09 -> кластеров: 2
eps=0.10 -> кластеров: 2
eps=0.10 -> кластеров: 2

Для 5 кластеров нужно eps = 0.07750000000000001


In [ ]:
eps_fixed = eps_for_5 

# Для каждого размера выборки от 0.1N до N
sample_sizes = np.arange(0.1, 1.01, 0.1) * N
sample_sizes = sample_sizes.astype(int)

avg_points_in_sphere = []

for size in sample_sizes:
    # Берем случайную подвыборку
    idx = np.random.choice(N, size=size, replace=False)
    X_sample = X[idx]
    
    nn = NearestNeighbors(radius=eps_fixed)
    nn.fit(X_sample)
    # Для каждой точки считаем сколько соседей 
    neighbors = nn.radius_neighbors(X_sample, return_distance=False)
    points_in_sphere = [len(n) for n in neighbors]
    
    avg_points = np.mean(points_in_sphere)
    avg_points_in_sphere.append(avg_points)
    
    print(f"n={size:4d} ({size/N*100:.0f}%): среднее точек в гиперсфере = {avg_points:.2f}")

print(f"\nИспользован eps = {eps_fixed}")

n=5000 (10%): среднее точек в гиперсфере = 3.35
n=10000 (20%): среднее точек в гиперсфере = 5.59
n=15000 (30%): среднее точек в гиперсфере = 7.96
n=20000 (40%): среднее точек в гиперсфере = 10.40
n=25000 (50%): среднее точек в гиперсфере = 12.70
n=30000 (60%): среднее точек в гиперсфере = 14.98
n=35000 (70%): среднее точек в гиперсфере = 17.45
n=40000 (80%): среднее точек в гиперсфере = 19.80
n=45000 (90%): среднее точек в гиперсфере = 22.10
n=50000 (100%): среднее точек в гиперсфере = 24.39

Использован eps = 0.07750000000000001


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=sample_sizes, y=avg_points_in_sphere,
                         mode='lines+markers',
                         marker=dict(size=10, color='blue', symbol='circle'),
                         line=dict(width=2, color='blue'),
                         name='Среднее число точек в ε-окрестности'))

fig.update_layout(title=f'Зависимость плотности точек от размера выборки<br>ε={eps_fixed:.3f}, целевое число кластеров=5',
                  xaxis_title='Количество точек n в выборке',
                  yaxis_title='Среднее количество точек в гиперсфере радиуса ε',
                  width=900, height=600,
                  font=dict(size=12),
                  xaxis=dict(tickvals=sample_sizes, tickangle=45),
                  yaxis=dict(gridcolor='lightgray', zerolinecolor='lightgray'))

fig.show()

print("\nСтатистика:")
for i, (size, avg) in enumerate(zip(sample_sizes, avg_points_in_sphere)):
    print(f"n={size:4d}: среднее = {avg:.2f} точек в гиперсфере")


Статистика:
n=5000: среднее = 3.35 точек в гиперсфере
n=10000: среднее = 5.59 точек в гиперсфере
n=15000: среднее = 7.96 точек в гиперсфере
n=20000: среднее = 10.40 точек в гиперсфере
n=25000: среднее = 12.70 точек в гиперсфере
n=30000: среднее = 14.98 точек в гиперсфере
n=35000: среднее = 17.45 точек в гиперсфере
n=40000: среднее = 19.80 точек в гиперсфере
n=45000: среднее = 22.10 точек в гиперсфере
n=50000: среднее = 24.39 точек в гиперсфере
